[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/capstone_template.ipynb) · [Course home](https://colab.research.google.com/github/stevenelliottjr/healthcare-analytics-course-data/blob/main/notebooks/00_course_home.ipynb)

**Google Colab version.** **Before you start: File → Save a copy in Drive.** Work only in your copy (its title starts with *Copy of*); changes to this original are not saved. Rename your copy `Capstone — <your last name>` (click the title at the top left).

**To submit (at each milestone):** **Runtime → Run all** and check that every cell ran, then **File → Save**. Click
**Share** (top right), add **steven.elliott@student.fairfield.edu** as **Editor**, and send. Colab keeps your saved versions
under **File → Revision history**; **File → Save and pin revision** keeps one permanently, which is how you
mark a submission.

**Coming back later?** Colab forgets variables and downloaded files when the runtime disconnects (after about 90 minutes idle). Open your copy and click **Runtime → Run all**.

# Capstone — *your question here*
**Author:** *(name)* · **Track:** quality and outcomes / place and equity / cost *(keep one)* ·
**Last updated:** *(date)* · **App:** *(link, from week 8)*

This is your semester project. You fill it in a section at a time, and each section is graded as a
milestone in the week shown. By week 14 it holds everything the final is graded on: the written brief,
the code and checks behind it, the link to your published app, and your evaluated AI analysis.

**How to use this template.** In week 2, save a copy in Drive (File → Save a copy in Drive) and rename it `Capstone — <your last name>`; your question goes in the title cell. Keep it one notebook all term; do not start a new one. Share it once with the instructor (steven.elliott@student.fairfield.edu, **Editor**); later saves reach the instructor on their own. Every time you submit a milestone, save and pin a revision (File → Save and pin revision) and write the milestone and date in the line below the title, so you and the instructor can see what changed (File → Revision history). Edit the markdown cells marked
*(edit)*; replace the `# TODO` code cells with your own. Delete these instruction paragraphs when you no
longer need them.

| Week | Milestone (2% each) | Section | Evidence the grader looks for |
|---:|---|---|---|
| 2 | Question brief | 1 | One page: track, question, why it matters, what data |
| 4 | Dataset documented | 2 | Data dictionary complete; `check()` written and passing |
| 6 | Analysis plan | 3 | Outcome, predictors, method, how success is judged |
| 8 | *(midterm)* App, version one | 5 | Reader-facing app notebook with an input, shared view-only (lab 8); link at the top of this notebook |
| 11 | Evaluation set | 6 | Ten questions with keys; the AI analyst's answers, grades and score |
| 13 | Draft brief | 7 | Full draft, reviewed by one peer; response-to-review table |
| 14 | **Final** (20%) | all | Brief, 10-minute presentation, published app, evaluated AI analysis |

### How the final is graded
| Part | Weight | What it means | Where in this notebook |
|---|---:|---|---|
| **Honest measurement** | 20% | Denominators, suppression and limitations are stated, not hidden. | Sections 2, 4, and the brief's *Data and measurement* and *Limitations* |
| **Analysis** | 25% | The method fits the question, and the evaluation is sound. | Sections 3 and 4 |
| **Information, not data** | 25% | The app and brief lead with what a decision-maker needs to know. A reader should not have to dig for the point. | Sections 5 and 7 |
| **AI work verified** | 15% | The AI analyst's answers were checked against a key, and you can say where it was wrong. | Section 6 |
| **Ownership** | 15% | You can explain any line of code in the notebook. | Section 8, and questions at the showcase |

In [ ]:
# --- Course setup: run this cell first -------------------------------------------
# Loaders for the public datasets this course uses. The canonical copy of this code
# is labs/course_data.py in the course repo; you will read it line by line in week 3.
import io, json, urllib.request, urllib.parse
from pathlib import Path
import pandas as pd
import duckdb

CACHE = Path("data"); CACHE.mkdir(exist_ok=True)

def sql(query: str) -> pd.DataFrame:
    """Run SQL over any DataFrame in this notebook by its variable name (DuckDB)."""
    return duckdb.sql(query).df()

def _get(url, params=None, timeout=120) -> bytes:
    if params:
        url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "healthcare-analytics-course/1.0"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return r.read()

# ---- CMS Care Compare (data.cms.gov/provider-data) --------------------------------
# File names change at every CMS refresh; dataset identifiers do not. We ask the
# catalog for the current CSV URL every time instead of hard-coding a file name.
CMS_CATALOG = "https://data.cms.gov/provider-data/api/1/metastore/schemas/dataset/items"
CMS_IDS = {
    "hospital_general_info": "xubh-q36u",   # one row per hospital
    "hrrp":                  "9n3s-kdb3",   # Hospital Readmissions Reduction Program, one row per hospital x measure
    "unplanned_visits":      "632h-zaca",   # readmissions / ED visits, one row per hospital x measure
    "complications_deaths":  "ynj2-r877",
    "timely_effective":      "yv7e-xc69",
}

def cms_csv_url(identifier: str) -> str:
    # Ask for the one dataset by id; the full keyword search can take minutes or time out.
    item = json.loads(_get(f"{CMS_CATALOG}/{identifier}", timeout=30))
    for d in item.get("distribution", []):
        if d.get("mediaType") == "text/csv":
            return d["downloadURL"]
    raise KeyError(f"No CSV distribution found for CMS dataset {identifier}")

# Course mirror: a snapshot of every CMS file the labs use, for when CMS is down or rate-limits
# a classroom downloading the same file at once. The live source is always tried first.
COURSE_DATA_BASE = "https://raw.githubusercontent.com/stevenelliottjr/healthcare-analytics-course-data/main/data"

def _fetch_or_mirror(path: Path, fetch, mirror_file: str) -> None:
    """Save `path` from the live source; if that fails, from the course mirror."""
    try:
        path.write_bytes(fetch())
    except Exception as e:
        print(f"Live download failed ({type(e).__name__}); using the course mirror for {path.name}.")
        path.write_bytes(_get(f"{COURSE_DATA_BASE}/{mirror_file}", timeout=300))

def load_cms(name: str, **read_csv_kwargs) -> pd.DataFrame:
    """Download (once) and read a Care Compare file. Everything is read as TEXT on purpose:
    IDs keep their leading zeros and 'Not Available' stays visible instead of silently
    becoming NaN. Converting columns to numbers is your job, and part of the lesson."""
    path = CACHE / f"cms_{name}.csv"
    if not path.exists():
        _fetch_or_mirror(path, lambda: _get(cms_csv_url(CMS_IDS[name]), timeout=300), f"cms/cms_{name}.csv")
    kw = dict(dtype=str, keep_default_na=False, encoding="utf-8-sig")
    kw.update(read_csv_kwargs)
    return pd.read_csv(path, **kw)

def to_num(s: pd.Series) -> pd.Series:
    """Text -> number; anything that is not a number ('Not Available', 'N/A', '') becomes NaN."""
    return pd.to_numeric(s, errors="coerce")

# ---- CDC PLACES, county release (Socrata API on data.cdc.gov) ---------------------
PLACES_COUNTY = "https://data.cdc.gov/resource/swc5-untb.json"
PLACES_SELECT = ("year,stateabbr,statedesc,locationname,locationid,categoryid,measureid,measure,"
                 "data_value,low_confidence_limit,high_confidence_limit,totalpopulation")

def load_places(measure_ids, data_value_type="Age-adjusted prevalence", state=None, page=50000) -> pd.DataFrame:
    """County-level model-based prevalence estimates (percent of adults). locationid is the
    5-digit county FIPS. Note: Connecticut reports *planning regions*, not counties, since 2022."""
    where = "measureid in(" + ",".join(f"'{m}'" for m in measure_ids) + f") AND data_value_type='{data_value_type}'"
    if state:
        where += f" AND stateabbr='{state}'"
    frames, offset = [], 0
    while True:
        chunk = pd.DataFrame(json.loads(_get(PLACES_COUNTY, {"$where": where, "$select": PLACES_SELECT,
                                                               "$limit": page, "$offset": offset})))
        frames.append(chunk)
        if len(chunk) < page:
            break
        offset += page
    df = pd.concat(frames, ignore_index=True)
    for c in ["data_value", "low_confidence_limit", "high_confidence_limit", "totalpopulation"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

# ---- Week 4 powerups (links checked 2026-10-26; 2022 is the latest SVI release) ----
SVI_COUNTY_URL = "https://svi.cdc.gov/Documents/Data/2022/csv/states_counties/SVI_2022_US_county.csv"
def load_svi_county() -> pd.DataFrame:
    """CDC/ATSDR Social Vulnerability Index 2022, one row per county.
    Key columns: FIPS, STATE, COUNTY, E_TOTPOP, RPL_THEMES (overall percentile 0-1),
    RPL_THEME1..RPL_THEME4. The value -999 means 'not available'."""
    path = CACHE / "svi_2022_county.csv"
    if not path.exists():
        path.write_bytes(_get(SVI_COUNTY_URL, timeout=300))
    return pd.read_csv(path, dtype={"FIPS": str}).replace(-999, pd.NA)

CHR_URL = "https://www.countyhealthrankings.org/sites/default/files/media/document/analytic_data2025_v3.csv"
def load_county_health_rankings() -> pd.DataFrame:
    """County Health Rankings analytic file. Row 0 holds readable labels and row 1 the variable
    codes; we skip the labels so the codes (fipscode, v001..v1xx) become the column names.
    The data dictionary on the CHR site maps each v-code to a measure."""
    path = CACHE / "chr_analytic_2025_v3.csv"
    if not path.exists():
        path.write_bytes(_get(CHR_URL, timeout=300))
    return pd.read_csv(path, skiprows=[0], dtype={"fipscode": str, "statecode": str, "countycode": str})

print("course setup ready")


## 1. Question brief *(milestone, week 2)*
About one page. Write for a reader who has never seen your data.

**Track:** *(edit)*

**The question, in one sentence.** Specific enough that a number could answer it. *(edit)*

**Who decides what with the answer?** Name the decision-maker (a role, not "policymakers") and the decision.
*(edit)*

**Why it matters.** Two or three sentences, with one number that shows the size of the problem. *(edit)*

**What data.** The dataset(s), who publishes them, the unit (hospital, county, claim) and the years.
*(edit)*

**What answer would change the decision?** If the result came out one way, what would they do; and the
other way? If nothing would change, pick a different question. *(edit)*

## 2. Dataset documented *(milestone, week 4)*
Load the data the way every lab does: as text, converted explicitly with `to_num`, so `Not Available`
never becomes a number by accident. Then document every column you will use, and write the checks that
would catch a bad refresh.

In [ ]:
# TODO: load your dataset(s). Keep the raw frame untouched; build working frames from it.
# raw = load_cms("hrrp")
# df = raw[raw["Measure Name"].eq("READM-30-HF-HRRP")].copy()
# df["excess_ratio"] = to_num(df["Excess Readmission Ratio"])


### Data dictionary
One row per column you use, including the ones you derive. *Missing / suppressed* records how the source
marks a value it does not report (`Not Available`, `Too Few to Report`, `-999`, blank) and how many rows
have it.

| Column | Type | Unit | Meaning, in plain words | Source (dataset, release date) | Missing / suppressed (code, count) | Notes |
|---|---|---|---|---|---|---|
| *e.g.* `Facility ID` | text | — | CMS certification number, one per hospital | CMS HRRP, *(release)* | none | key; 6 characters, keep leading zeros |
| *e.g.* `excess_ratio` | number | ratio | Predicted / expected readmissions | derived from `Excess Readmission Ratio` | `N/A`: *(count)* | > 1 means penalty range |
| | | | | | | |

In [ ]:
def check(df: pd.DataFrame) -> pd.DataFrame:
    """Validation checks for my capstone data, in the week 3 pattern: one row per check, with the number
    of failing rows and one example. Run it after every reload. Every check should show 0 failures,
    and any that does not is either a data problem to fix or a fact to report in the brief."""
    rows = []
    def add(name, mask, col):
        bad = df[mask]
        rows.append({"check": name, "failures": int(mask.sum()), "example": bad[col].iloc[0] if len(bad) else ""})
    # TODO: one add(...) per check. Cover at least:
    #   - the key is unique (and has the right format)
    #   - required columns have no unexpected blanks
    #   - each numeric column: how many values failed to_num (report them; do not fill them with 0)
    #   - values sit in a plausible range (a rate between 0 and 100, a ratio above 0)
    #   - the row count is what the source documentation says
    # Example:
    # add("facility id unique", df["Facility ID"].duplicated(keep=False), "Facility ID")
    return pd.DataFrame(rows)

# check(df)


## 3. Analysis plan *(milestone, week 6)*
About one page. Write it **before** you fit anything; you will be held to it, or asked why you changed it.

**Outcome.** What you measure, its exact definition, the unit and the denominator. *(edit)*

**Predictors or comparison groups.** What, and why each one. Mark any that could leak the outcome. *(edit)*

**Method.** The model or comparison, and the simplest baseline it has to beat. *(edit)*

**How success is judged.** The metric(s) and the threshold that would count as useful, decided now.
For a model: AUROC / AUPRC / calibration on a hold-out. For a comparison: an effect size with its
uncertainty. *(edit)*

**Subgroups.** Which groups you will check separately, and why. *(edit)*

**What could go wrong.** Small numbers, suppression, missing data, confounding. *(edit)*

## 4. Analysis
The work itself. Keep the cells in the order of the plan, give each a short label, and put a one-line
markdown note above any cell whose purpose is not obvious. If you publish any **count table**, run it
through the suppression check from lab 12 first (copy `lines`, `recover` and `check` from lab 12 into
one cell and rename that `check` to `suppression_check` so it does not replace your data `check()`).

In [ ]:
# TODO 4a: the outcome, with its denominator. Report how many units are in, and how many were dropped and why.


In [ ]:
# TODO 4b: the baseline first, then your method.


In [ ]:
# TODO 4c: evaluation as planned in section 3: hold-out metrics or effect size with uncertainty; subgroups.


**What the analysis found**, in two to four sentences with numbers. This becomes the brief's
*What we found*. *(edit)*

## 5. The app *(midterm, week 8; final, week 14)*
**Published app:** *(link)*

**Inputs, and what each one changes:** *(edit)*

**What a reader should see first:** the one sentence the default view answers. *(edit)*

In Colab the app is a separate, reader-facing notebook built the way lab 8 shows: a `# @param` dropdown, a plotly chart, a headline sentence, every code cell hidden, shared with a view-only link. Copy into it the cells it needs from this notebook.

## 6. Evaluation set *(milestone, week 11)*
Ten questions a decision-maker might ask the AI analyst about **your** data, with answer keys **you**
computed in this notebook. Mix easy lookups with questions that need a denominator, a filter or a join,
and include at least one the data cannot answer (the right response is to say so).

| # | Question | Answer key | Key computed in cell | AI analyst's answer | Grade (right / partly / wrong) | What went wrong |
|---:|---|---|---|---|---|---|
| 1 | | | | | | |
| 2 | | | | | | |
| 3 | | | | | | |
| 4 | | | | | | |
| 5 | | | | | | |
| 6 | | | | | | |
| 7 | | | | | | |
| 8 | | | | | | |
| 9 | | | | | | |
| 10 | | | | | | |

**Score:** *x* / 10 right, *y* partly. **The failure that matters most, and why:** *(edit)*

In [ ]:
# TODO: one small, labelled computation per answer key, so the grader can rerun it.
# key_1 = ...


## 7. The brief *(draft, week 13; final, week 14)*
The written brief, in this cell or the cells below it. Write for the decision-maker in section 1: lead
with the answer, then the evidence. Replace each prompt with your text; keep the headings.

### *Title that states the finding*
**Bottom line.** Three sentences: what you found, how sure you are, what the reader should do.

**The question and why it matters.** From section 1, tightened.

**Data and measurement.** Source, unit, years, denominators, what was suppressed or missing and how many.

**What we found.** Two or three findings, each with a number and one chart whose title states its point.

**How sure we are.** The evaluation from section 4c in plain words, with uncertainty and subgroups.

**What the AI analyst got right and wrong.** The score from section 6 and the failure that matters.

**Limitations.** What the data cannot see; what would change the answer.

**Recommendation and next step.**

*Appendix: link to the app, this notebook, and the data dictionary.*

### Response to peer review *(week 13)*
| Suggestion | Will change? (yes / no) | What I will do, or why not |
|---|---|---|
| | | |

## 8. AI use log *(ownership)*
"AI may write the code; you own it." Keep this log as you go. It is not a penalty; it is how you show
you checked.

| Date | Cell or section | What the assistant did | What you changed | How you verified it |
|---|---|---|---|---|
| | | | | |

**Before the showcase:** open each code cell and make sure you could explain every line out loud. The
week 14 checklist (lab 14) lists everything the final submission needs.